# rhino-poc — Colab

**Baslamadan:** Runtime > Change runtime type > **GPU (T4)**.

Hucreleri **sirayla** kos. Atlama — 2. hucre degiskenleri tanimlar,
sonraki hucreler onlari kullanir (`NameError` alirsan 2. hucreyi kosmamissindir).

Colab burada sadece **GPU gereken adim** icin: COLMAP MVS. Bkz. `PLAN.md` SS4.

In [1]:
!nvidia-smi

Wed Aug 12 07:50:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Kurulum — repo + Drive + yollar
Tek hucre, her seyi tanimlar. Oturum dusarse **bu hucreden** devam et.

In [2]:
import os

REPO = '/content/rhino-poc'
WORK = '/content/work'          # calisma diski (HIZLI)

if os.path.isdir(REPO + '/.git'):
    !cd {REPO} && git fetch --quiet origin && git reset --hard origin/main
else:
    !rm -rf {REPO}
    !git clone --quiet https://github.com/Daml4Yilmaz/rhino-poc.git {REPO}

from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/rhino-poc-data'   # GIRDI: video/yakalama
SAVE = '/content/drive/MyDrive/rhino-poc-out'    # CIKTI: mesh buraya kaydedilir
os.makedirs(DATA, exist_ok=True)
os.makedirs(SAVE, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

!cd {REPO} && git log --oneline -1
print('DATA icerigi:', os.listdir(DATA))

HEAD is now at 20733b9 Fail loudly on a missing capture path instead of falling through to ARKit
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
20733b9 (HEAD -> main, origin/main, origin/HEAD) Fail loudly on a missing capture path instead of falling through to ARKit
DATA icerigi: ['vaka_001', 'test.MOV']


**Neden `/content/work`?** COLMAP MVS on binlerce kucuk dosya yazar.
Drive FUSE uzerinden bu cok yavastir ve kopabilir. Hesaplama hizli diskte kosar,
**sonuclar** son hucrede Drive'a kopyalanir.

## 2. COLMAP (CUDA'li)
`%%bash` hucresi Python degiskenlerini GOREMEZ — burada yol yazmiyoruz, sorun degil.

In [3]:
%%bash
cd /opt
if [ ! -x /opt/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
if [ ! -x /opt/colmapenv/bin/colmap ]; then
  /opt/bin/micromamba create -y -q -p /opt/colmapenv -c conda-forge 'colmap=*=gpu*'
fi

In [3]:
import os
for c in ('/opt/colmapenv/bin/colmap', '/usr/local/colmap/bin/colmap'):
    if os.path.exists(c):
        os.system('ln -sf ' + c + ' /usr/local/bin/colmap')
        break
!colmap -h 2>&1 | head -3

COLMAP 3.11.1 -- Structure-from-Motion and Multi-View Stereo
(Commit Unknown on Unknown with CUDA)



Ciktida **`with CUDA`** yazmali. `without CUDA` yazarsa MVS adimi kosmaz.

**PATH uyarisi:** conda klasorunu PATH'in basina EKLEME — icindeki python sistem
python'unu golgeler ve `cv2` kirilir. Sadece symlink (ustteki hucre bunu yapiyor).

## 3. Python paketleri

In [4]:
# OpenCV KURMUYORUZ: Colab'da zaten var. Ustune opencv-contrib-python
# kurulunca cv2 namespace'i yarim yukleniyor (SIFT_create calisir,
# CascadeClassifier kaybolur). ArUco birakildigi icin contrib gereksiz.
!pip install -q open3d trimesh pycolmap typer pandas pillow
!pip install -q --force-reinstall --no-deps {REPO}

import cv2
assert hasattr(cv2, 'CascadeClassifier') and hasattr(cv2, 'SIFT_create'), \
    'cv2 yarim yuklendi - opencv paketleri catisiyor'
print('cv2', cv2.__version__, 'saglam')
!python -m poc.cli --help | head -12

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
                                                                                
 Usage: python -m poc.cli [OPTIONS] COMMAND [ARGS]...                           
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ process                                                                      │
│ scale     Sadece olcek: ARKit pozu + LiDAR -> scale.json                     │
│ measure   Sadece olcum: landmarks.json -> measurements.json                  │
╰──────────────────────────────────────────────────────

`poc` yerine **`python -m poc.cli`** kullaniyoruz: kurulan komut bazen Colab'in
PATH'ine girmez, modul cagrisi her zaman calisir.

**`-e` (editable) KULLANMA** — Jupyter ayni oturumda import edemez.
Repoda kod degisirse 1. ve bu hucreyi tekrar kos.

## 4A. TEST — duz .mov / .mp4

Videoyu Drive'a koy: `MyDrive/rhino-poc-data/test.mov`

Sadece **fotogrametri hattini** dogrular: kareler -> SfM -> MVS -> mesh.
**Model BIRIMSIZ olur** (`model_unitless.glb`) — aci ve Goode orani anlamli,
mm cinsinden uzunluk/genislik/sapma URETILEMEZ. Onun icin 4B gerekir.

Ilk denemede `--n-frames 150` birak: MVS kare sayisiyla dogru orantili,
150 kare T4'te ~15-25 dk, 300 kare bir saati asabilir.

In [7]:
VIDEO = DATA + '/test.mov'
OUT   = WORK + '/test_out'
!python -m poc.cli process {VIDEO} --out {OUT} --n-frames 150 --max-dim 1600

[poc] TEST MODU: duz video — ARKit pozu yok, LiDAR yok.
[poc] Fotogrametri (frames+sfm+mvs) kosar, OLCEK KOSMAZ.
[poc] Cikan model BIRIMSIZ olur: acilar ve Goode orani anlamli,
[poc] mm cinsinden uzunluk/genislik/sapma URETILEMEZ.
[frames] 5146 kaynak kare -> 150 secildi (medyan keskinlik 237, esik 40)
[sfm] 3 kare kaydedildi, 271 nokta
[mvs] mesh: 20559 vertex -> /content/work/test_out/mesh_raw.ply
[scale] TEST MODU — atlandi. Model birimsiz kalacak.
[export] /content/work/test_out/model_unitless.glb  (bbox mm: 25 x 32 x 19)
[measure] landmarks.json yok — atlandi (hafta 2: FLAME kaydi uretecek).
[poc] bitti — 1536 sn


## 4B. GERCEK — Stray Scanner yakalamasi

Klasoru oldugu gibi Drive'a kopyala: `MyDrive/rhino-poc-data/vaka_001_stray/`
icinde `rgb.mp4`, `odometry.csv`, `camera_matrix.csv`, `depth/`, `confidence/`.

Bu yolda olcek de kosar. Beklenen: `agreement_pct` < 1.5, `scale_verified: true`.

In [ ]:
CAPTURE = DATA + '/vaka_001_stray'
OUT     = WORK + '/vaka_001'
!python -m poc.cli process {CAPTURE} --out {OUT} --n-frames 300

## 5. Sonuclari Drive'a kaydet
Yukarida hangi hucreyi kostuysan (`OUT` ondan gelir) bunu kos.

In [ ]:
import os, shutil, json

case = os.path.basename(OUT)
dst  = os.path.join(SAVE, case)
os.makedirs(dst, exist_ok=True)

KEEP = ['mesh_raw.ply', 'model.glb', 'model_unitless.glb',
        'scale.json', 'measurements.json', 'frames_index.json',
        'landmarks.json']
for f in KEEP:
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        shutil.copy2(p, dst)
        print('kaydedildi: %-22s %8.1f MB' % (f, os.path.getsize(p)/1e6))

print('\nDrive konumu:', dst)
sj = os.path.join(dst, 'scale.json')
if os.path.exists(sj):
    print(json.dumps(json.load(open(sj)), indent=2))

`mesh_raw.ply` ham yuzey, `model.glb` / `model_unitless.glb` goruntuleyici icin.
GLB'yi [gltf.report](https://gltf.report) veya Blender'da ac.
4B'de kafa bbox ~200-250 mm cikmali.

COLMAP ara ciktilari (`colmap/`, `frames/`) bilerek kopyalanmaz — GB'larca yer tutar.
Gerekirse: `!cp -r {OUT}/colmap {dst}/`

In [ ]:
import os, cv2, sqlite3, pycolmap

# 1) Video gercekte ne?
c = cv2.VideoCapture(VIDEO)
fps = c.get(cv2.CAP_PROP_FPS) or 1
n   = c.get(cv2.CAP_PROP_FRAME_COUNT)
print(f"video: {int(c.get(3))}x{int(c.get(4))}, {fps:.0f} fps, "
      f"{int(n)} kare, {n/fps:.0f} saniye")
c.release()

# 2) COLMAP kac alt-model uretti? (eski kod hep 0'i aliyordu)
sp = OUT + '/colmap/sparse'
for d in sorted(os.listdir(sp)):
    p = os.path.join(sp, d)
    try:
        r = pycolmap.Reconstruction(p)
        print(f"  sparse/{d}: {r.num_reg_images()} kare, {r.num_points3D()} nokta")
    except Exception as e:
        print(f"  sparse/{d}: okunamadi ({e})")

# 3) Doku ve eslesme var mi? (asil teshis)
con = sqlite3.connect(OUT + '/colmap/database.db')
ni, avg = con.execute("SELECT COUNT(*), AVG(rows) FROM keypoints").fetchone()
print(f"\nSIFT: {ni} goruntu, kare basina ortalama {avg:.0f} ozellik")
tot = con.execute("SELECT COUNT(*) FROM two_view_geometries").fetchone()[0]
ok  = con.execute("SELECT COUNT(*) FROM two_view_geometries WHERE rows >= 15").fetchone()[0]
print(f"eslesme: {ok}/{tot} kare cifti gecerli geometri verdi")
con.close()
